# 🧹 PHASE 2: DATA PREPARATION
## Data Cleaning, Validation & Aggregation

**Objective:** Clean, validate, and aggregate data for feature engineering

**Input:** Raw data (Documents1.csv)  
**Output:** Cleaned & aggregated dataset ready for feature engineering

---

In [ ]:
# Set working directory to project root
import os
from pathlib import Path

# Get notebook directory and navigate to project root
notebook_dir = Path.cwd()
project_root = notebook_dir.parent.parent
os.chdir(project_root)
print(f"Working directory set to: {os.getcwd()}")

# Imports
import pandas as pd
import numpy as np
import sys

sys.path.insert(0, 'src/scripts')
from config import RAW_DATA_FILE, PROCESSED_DATA_DIR
from logger import get_logger

logger = get_logger(__name__)

print("✅ Imports successful!")

Working directory set to: c:\Users\1boughai\Desktop\IDP-Monitoring-Project
2026-05-20 16:00:09,054 - logger - INFO - 🔧 DEBUG MODE ENABLED
2026-05-20 16:00:09,062 - logger - INFO - 📝 Logs saved to: c:\Users\1boughai\Desktop\IDP-Monitoring-Project\src\outputs\project_20260520_160009.log
✅ Imports successful!


## 1️⃣ LOAD RAW DATA

In [6]:
# Load data
logger.info(f"Loading raw data from {RAW_DATA_FILE}")
df = pd.read_csv(RAW_DATA_FILE, low_memory=False)

print(f"✅ Loaded {len(df):,} rows × {df.shape[1]} columns")
print(f"\n📊 Before cleaning: {len(df):,} transactions")

✅ Loaded 616,800 rows × 59 columns

📊 Before cleaning: 616,800 transactions


## 2️⃣ FILTER VALID TRANSACTIONS

In [3]:
# Remove deleted items
initial_count = len(df)

if 'deletion_indicator_|_loekz' in df.columns:
    df = df[df['deletion_indicator_|_loekz'].isna()]
    logger.info(f"Removed {initial_count - len(df):,} deleted items")

# Remove zero/null amounts
initial_count = len(df)
df = df[df['amount_|_wrbtr'].notna()]
df = df[df['amount_|_wrbtr'] != 0]
logger.info(f"Removed {initial_count - len(df):,} zero-amount items")

# Filter valid P2P categories
valid_categories = ['E', 'Q']  # E=GR, Q=IR
initial_count = len(df)
df = df[df['po_history_category_|_bewtp'].isin(valid_categories)]
logger.info(f"Kept only {valid_categories} categories: {initial_count - len(df):,} removed")

print(f"\n✅ After filtering: {len(df):,} transactions")


✅ After filtering: 605,742 transactions


## 3️⃣ HANDLE MISSING VALUES

In [4]:
# Fill missing supplier
if 'supplier_|_lifnr' in df.columns:
    df['supplier_|_lifnr'] = df['supplier_|_lifnr'].fillna('UNKNOWN')

# Fill missing supplier name
if 'supplier_name_|_name1' in df.columns:
    df['supplier_name_|_name1'] = df['supplier_name_|_name1'].fillna('UNKNOWN SUPPLIER')

# Fill missing invoice value
if 'invoice_value_|_reewr' in df.columns:
    df['invoice_value_|_reewr'] = df['invoice_value_|_reewr'].fillna(0.0)

print("✅ Missing values handled")
print(f"\nRemaining nulls in critical columns:")
critical_cols = ['purchasing_document_|_ebeln', 'item_|_ebelp', 'amount_|_wrbtr']
for col in critical_cols:
    if col in df.columns:
        nulls = df[col].isna().sum()
        if nulls > 0:
            print(f"  {col}: {nulls}")

✅ Missing values handled

Remaining nulls in critical columns:


## 4️⃣ AGGREGATE BY (PO, ITEM)

In [ ]:
# Key aggregation: group by (PO, Item)
print("Aggregating by (PO, Item)...\n")

# Create PO+Item key
df['po_item_key'] = df['purchasing_document_|_ebeln'].astype(str) + '_' + df['item_|_ebelp'].astype(str)

# For each PO+Item, aggregate:
# - Keep one record per PO+Item
# - Sum amounts for GR and IR separately
# - Take first supplier

agg_dict = {
    'purchasing_document_|_ebeln': 'first',
    'item_|_ebelp': 'first',
    'supplier_|_lifnr': 'first',
    'po_history_category_|_bewtp': lambda x: '|'.join(x.unique()),  # GR and/or IR
    'amount_|_wrbtr': 'sum',  # Total amount
    'invoice_value_|_reewr': 'sum',  # Total invoice
}

# Add supplier name if it exists
if 'supplier_name_|_name1' in df.columns:
    agg_dict['supplier_name_|_name1'] = 'first'

# Handle date columns
date_cols = [col for col in df.columns if 'date' in col.lower()]
for col in date_cols:
    if col in df.columns:
        agg_dict[col] = 'first'

# Add other columns
for col in df.columns:
    if col not in agg_dict and col != 'po_item_key':
        if col in df.columns:
            agg_dict[col] = 'first'

# Aggregate
df_agg = df.groupby('po_item_key').agg(agg_dict).reset_index()

print(f"✅ Aggregated {len(df):,} rows → {len(df_agg):,} unique PO+Items")
print(f"   Compression ratio: {len(df)/len(df_agg):.1f}x")

Aggregating by (PO, Item)...

✅ Aggregated 616,800 rows → 295,651 unique PO+Items
   Compression ratio: 2.1x


## 5️⃣ DETECT OUTLIERS & ANOMALIES

In [9]:
# Calculate amount gap
df_agg['amount_gap'] = abs(df_agg['amount_|_wrbtr'] - df_agg['invoice_value_|_reewr'])
df_agg['amount_gap_pct'] = (df_agg['amount_gap'] / (df_agg['amount_|_wrbtr'] + 1)) * 100

# Flag outliers (amount differences)
df_agg['has_large_gap'] = df_agg['amount_gap_pct'] > 10  # >10% difference

# Detect GR/IR combinations
df_agg['has_gr'] = df_agg['po_history_category_|_bewtp'].str.contains('E', na=False)
df_agg['has_ir'] = df_agg['po_history_category_|_bewtp'].str.contains('Q', na=False)

print("⚠️ OUTLIERS & ANOMALIES:")
print(f"  Large amount gaps (>10%): {df_agg['has_large_gap'].sum():,}")
print(f"  GR without IR: {(df_agg['has_gr'] & ~df_agg['has_ir']).sum():,}")
print(f"  IR without GR (FRAUD!): {(~df_agg['has_gr'] & df_agg['has_ir']).sum():,}")
print(f"  Neither GR nor IR: {(~df_agg['has_gr'] & ~df_agg['has_ir']).sum():,}")

⚠️ OUTLIERS & ANOMALIES:
  Large amount gaps (>10%): 290,173
  GR without IR: 12,624
  IR without GR (FRAUD!): 0
  Neither GR nor IR: 0


## 6️⃣ DATA QUALITY CHECK

In [10]:
# Final quality check
print("\n✅ FINAL DATA QUALITY:")
print(f"  Total records: {len(df_agg):,}")
print(f"  Duplicates: {df_agg.duplicated().sum()}")
print(f"  Null PO: {df_agg['purchasing_document_|_ebeln'].isna().sum()}")
print(f"  Null Item: {df_agg['item_|_ebelp'].isna().sum()}")
print(f"  Null supplier: {df_agg['supplier_|_lifnr'].isna().sum()}")

# Validate PO+Item uniqueness
po_item_unique = df_agg[['purchasing_document_|_ebeln', 'item_|_ebelp']].drop_duplicates()
print(f"\n  Unique PO+Items: {len(po_item_unique):,}")
print(f"  Consistency: {len(po_item_unique) == len(df_agg)}")


✅ FINAL DATA QUALITY:
  Total records: 295,651
  Duplicates: 0
  Null PO: 0
  Null Item: 0
  Null supplier: 0

  Unique PO+Items: 295,651
  Consistency: True


## 7️⃣ EXPORT PREPARED DATA

In [11]:
# Create output directory
Path(PROCESSED_DATA_DIR).mkdir(parents=True, exist_ok=True)

# Save prepared data
output_file = f"{PROCESSED_DATA_DIR}/data_prepared_phase2.csv"
df_agg.to_csv(output_file, index=False)
logger.info(f"✅ Saved prepared data to {output_file}")

print(f"\n📊 PREPARED DATA SUMMARY:")
print(f"  Shape: {df_agg.shape}")
print(f"  Columns: {df_agg.shape[1]}")
print(f"  Saved: {output_file}")
print(f"\n✅ Data Preparation Complete!")


📊 PREPARED DATA SUMMARY:
  Shape: (295651, 65)
  Columns: 65
  Saved: c:\Users\1boughai\Desktop\IDP-Monitoring-Project\src\data\processed/data_prepared_phase2.csv

✅ Data Preparation Complete!


## ✅ PHASE 2 SUMMARY

**Actions Completed:**
- ✅ Removed deleted items
- ✅ Removed zero/null amounts
- ✅ Kept only valid P2P categories (GR, IR)
- ✅ Handled missing values
- ✅ Aggregated by (PO, Item)
- ✅ Detected outliers & anomalies
- ✅ Validated data quality
- ✅ Exported prepared dataset

**Next Step:** Proceed to 04_feature_engineering.ipynb